## IMPORTS

In [1]:
import os
import sys
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from PIL import Image
from supervision import Detections
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## CROP FACES FROM PHOTOS

In [2]:



# 1. Settings
padding_ratio = 0.15   # 15% padding around the box

# Confidence threshold to ignore weak detections
conf_threshold = 0.25

# Supported image extensions
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# 2. Load YOLO face model

model_path = hf_hub_download(
    repo_id="arnabdhar/YOLOv8-Face-Detection",
    filename="model.pt"
)
cropping_model = YOLO(model_path)


# 3. Helper functions

def choose_best_face(boxes, confs):
    """
    Choose the best face using area * confidence.
    Prefers large, confident detections.
    """
    best_idx = None
    best_score = -1

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        w = max(0, x2 - x1)
        h = max(0, y2 - y1)
        area = w * h
        conf = confs[i] if confs is not None else 0.6
        score = area * conf

        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx


def add_padding_and_clip(x1, y1, x2, y2, img_w, img_h, pad_ratio=0.15):
    """
    Add padding to a bounding box and clip it to image boundaries.
    """
    w = x2 - x1
    h = y2 - y1

    pad_x = int(w * pad_ratio)
    pad_y = int(h * pad_ratio)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(img_w, x2 + pad_x)
    y2 = min(img_h, y2 + pad_y)

    return x1, y1, x2, y2


def process_image(image_input, cropping_model, conf_threshold=0.25, padding_ratio=0.15):
    """
    Detect faces in one image.
    - If no face and input is a file path: delete image
    - If faces found: choose best one and crop
    Accepts:
    - image path: str or Path
    - OpenCV image/frame: np.ndarray
    - PIL image: Image.Image

    Returns:
        cropped_pil, message
    """

    image_path = None

    try:
        # Image file path
        if isinstance(image_input, (str, Path)):
            image_path = Path(image_input)

            pil_img = Image.open(image_path).convert("RGB")

            # Read with cv2 for cropping/saving
            image = cv2.imread(str(image_path))

            if image is None:
                return None, "ERROR reading image with cv2"

        # OpenCV image / video frame
        elif isinstance(image_input, np.ndarray):
            image = image_input.copy()

            pil_img = Image.fromarray(
                cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            )

        # PIL image
        elif isinstance(image_input, Image.Image):
            pil_img = image_input.convert("RGB")

            image = cv2.cvtColor(
                np.array(pil_img),
                cv2.COLOR_RGB2BGR
            )

        else:
            return None, "ERROR unsupported image type"

    except Exception as e:
        return None, f"ERROR opening image: {e}"

    # Run inference
    results = cropping_model(pil_img, verbose=False)
    result = results[0]

    img_h, img_w = image.shape[:2]

    # Extract boxes/confidences
    if result.boxes is None or len(result.boxes) == 0:
        if image_path is not None:
            os.remove(image_path)
        return None, "DELETED - no face"

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy() if result.boxes.conf is not None else None

    # Filter by confidence threshold
    filtered_boxes = []
    filtered_confs = []

    for i, box in enumerate(boxes):
        conf = confs[i] if confs is not None else 1.0
        if conf >= conf_threshold:
            filtered_boxes.append(box)
            filtered_confs.append(conf)

    if len(filtered_boxes) == 0:
        if image_path is not None:
            os.remove(image_path)
        return None, "DELETED - no face above confidence threshold"

    filtered_boxes = np.array(filtered_boxes)
    filtered_confs = np.array(filtered_confs)

    # Choose best face
    best_idx = choose_best_face(filtered_boxes, filtered_confs)
    x1, y1, x2, y2 = filtered_boxes[best_idx]

    # Convert to ints
    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

    # Add padding and keep inside image
    x1, y1, x2, y2 = add_padding_and_clip(
        x1, y1, x2, y2, img_w, img_h, pad_ratio=padding_ratio
    )

    # Validate crop
    if x2 <= x1 or y2 <= y1:
        if image_path is not None:
            os.remove(image_path)
        return None, "DELETED - invalid crop"

    cropped = image[y1:y2, x1:x2]

    if cropped.size == 0:
        if image_path is not None:
            os.remove(image_path)
        return None, "DELETED - empty crop"

    # Convert OpenCV (BGR) → PIL (RGB)
    cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    cropped_pil = Image.fromarray(cropped_rgb)

    best_conf = filtered_confs[best_idx]
    message = f"CROPPED - conf={best_conf:.3f}"

    return cropped_pil, message


## MODEL

In [3]:
import torch
import torch.nn as nn
from torchvision import models

class EmotionResNet(nn.Module):
    def __init__(self, num_classes, freeze_backbone=False):
        super().__init__()

        # Switch to ResNet34
        self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

        in_features = self.backbone.fc.in_features

        # Improved head
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

        # Optional freezing
        if freeze_backbone:
            for name, param in self.backbone.named_parameters():
                if not name.startswith("fc"):
                    param.requires_grad = False

    def forward(self, x):
        return self.backbone(x)

## TRANSFORMS

In [4]:
from torchvision import transforms

res_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Extract frames and preprocess video

In [6]:
import cv2

def extract_frames(video_path):
    
    cap = cv2.VideoCapture(video_path)
    
    frames = []
    
    success = True
    
    while success:
    
        success, frame = cap.read()
    
        if success:
            frames.append(frame)
    
    cap.release()
    
    print(f"{len(frames)} frames extracted!")
    return frames

## General Purpose image classifier

In [9]:
from PIL import Image
import torch
import numpy as np

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CLASS_NAMES = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

model = EmotionResNet(
    num_classes=7,
    freeze_backbone=False
)


model.to(DEVICE)
model.eval()
model.load_state_dict(torch.load("C:/Users/User/OneDrive/Documents/resNET emotion recognition/best_resnet34.pth", map_location=device)) #LOADS BEST MODEL

def classify_image(image_input):


    # If input is a file path
    if isinstance(image_input, str):

        img = Image.open(
            image_input
        ).convert("RGB")
        
    # If input is already an image/frame

    # If input is an OpenCV frame / numpy array
    elif isinstance(image_input, np.ndarray):
    
        img = Image.fromarray(
            cv2.cvtColor(
                image_input,
                cv2.COLOR_BGR2RGB
            )
        )
    else:
            img = image_input.convert("RGB")

    # Transform
    input_tensor = (
        res_val_transform(img)
        .unsqueeze(0)
        .to(DEVICE)
    )

    # Predict
    with torch.no_grad():

        outputs = model(input_tensor)

        probs = torch.softmax(
            outputs,
            dim=1
        )

    return probs.cpu().numpy()

## Create annotated videos with smoothed sentement predictions

In [10]:
from tqdm.notebook import tqdm
import numpy as np
import cv2
import os
from moviepy.editor import VideoFileClip

def classify_video(video_path):

    frames = extract_frames(video_path)

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    processed_video = []

    output_no_audio = "processed_no_audio.mp4"
    final_output = "processed_with_audio.mp4"

    out = cv2.VideoWriter(
        output_no_audio,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    # MAIN VIDEO PROGRESS BAR
    video_pbar = tqdm(
        total=len(frames),
        desc="Processing Video",
        position=0,
        leave=True
    )

    for i in range(len(frames)):

        current_frame = frames[i].copy()

        probs = []

        window = frames[
            max(0, i-5):min(len(frames), i+6)
        ]

        # WINDOW PROGRESS BAR
        window_pbar = tqdm(
            total=len(window),
            desc=f"Window {i+1}/{len(frames)}",
            position=1,
            leave=False
        )

        for frame_idx, w_frame in enumerate(window):

            # Crop face
            cropped_face, crop_message = process_image(
                w_frame,
                cropping_model,
                conf_threshold=0.25,
                padding_ratio=0.15
            )

            # Skip failed crops
            if cropped_face is None:
                window_pbar.update(1)
                continue

            # Classify
            frame_probs = classify_image(cropped_face)

            probs.append(frame_probs)

            # Convert shape (1, num_classes) -> (num_classes,)
            frame_probs = frame_probs.squeeze()

            pred_idx = np.argmax(frame_probs)

            window_pbar.update(1)

        window_pbar.close()

        # Skip if no valid predictions
        if len(probs) == 0:
            out.write(current_frame)
            video_pbar.update(1)
            continue

        # Average probabilities across window
        probs = np.vstack(probs)

        means = probs.mean(axis=0)

        pred = np.argmax(means)
        confidence = means[pred]

        cv2.putText(
            current_frame,
            f"{CLASS_NAMES[pred]} Conf: {confidence:.3f}",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        processed_video.append(current_frame)

        out.write(current_frame)

        video_pbar.update(1)

    video_pbar.close()

    out.release()

    original_video_path = video_path

    print("\nAdding audio back to video...")

    processed_clip = VideoFileClip(output_no_audio)
    original_clip = VideoFileClip(original_video_path)

    final_clip = processed_clip.set_audio(original_clip.audio)

    final_clip.write_videofile(
        final_output,
        codec="libx264",
        audio_codec="aac"
    )

    processed_clip.close()
    original_clip.close()
    final_clip.close()

    print("\nDone!")

    return os.path.abspath(final_output)

## APP WHICH PROCESSES EITHER PHOTOS OF VIDEO

In [ ]:
import tkinter as tk
from tkinter import Label, Button
from PIL import Image, ImageTk
from tkinterdnd2 import DND_FILES, TkinterDnD
import os
import torch


class ImageDropApp:
    def __init__(self, root, face_model):
        self.root = root
        self.face_model = face_model
        self.root.title("Drag and Drop Image or Video")
        self.root.geometry("600x750")

        self.image_path = None
        self.video_path = None

        # Drop area
        self.drop_label = Label(
            root,
            text="Drag and drop an image or video here",
            relief="ridge",
            bd=2,
            bg="lightgray"
        )
        self.drop_label.pack(padx=20, pady=10, fill="both", expand=True)

        self.drop_label.drop_target_register(DND_FILES)
        self.drop_label.dnd_bind("<<Drop>>", self.drop_file)

        # Original image
        self.original_label = Label(root, text="Original Image")
        self.original_label.pack()

        self.original_image_label = Label(root)
        self.original_image_label.pack(pady=5)

        # Analyse button
        self.analyse_button = Button(
            root,
            text="Analyse Image",
            command=self.analyse_image,
            bg="blue",
            fg="white"
        )
        self.analyse_button.pack(pady=10)

        # Processed image
        self.result_label = Label(root, text="Processed Image")
        self.result_label.pack()

        self.result_image_label = Label(root)
        self.result_image_label.pack(pady=5)

        # Prediction text
        self.prediction_label = Label(
            root,
            text="Prediction: None",
            font=("Arial", 14, "bold")
        )
        self.prediction_label.pack(pady=10)

        self.tk_original = None
        self.tk_result = None

    # Handle drop
    def drop_file(self, event):
        file_path = event.data.strip("{}")

        image_exts = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
        video_exts = (
            ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv",
            ".webm", ".mpeg", ".mpg", ".m4v", ".3gp"
        )

        if file_path.lower().endswith(image_exts):
            self.image_path = file_path
            self.video_path = None

            image = Image.open(file_path).convert("RGB")
            image.thumbnail((400, 400))
            image = image.copy()

            self.tk_original = ImageTk.PhotoImage(image)

            self.original_image_label.config(image=self.tk_original)
            self.original_image_label.image = self.tk_original

            self.result_image_label.config(image="")
            self.result_image_label.image = None

            self.drop_label.config(text=os.path.basename(file_path))
            self.prediction_label.config(text="Prediction: None")

        elif file_path.lower().endswith(video_exts):
            self.video_path = file_path
            self.image_path = None

            self.original_image_label.config(image="")
            self.original_image_label.image = None

            self.result_image_label.config(image="")
            self.result_image_label.image = None

            self.drop_label.config(text="Analysing video... please wait")
            self.prediction_label.config(text="Video analysis running...")
            self.root.update()

            analysed_video = classify_video(file_path)

            self.drop_label.config(
                text=f"Analysed video: {os.path.basename(analysed_video)}"
            )
            self.prediction_label.config(text="Video analysis complete")

            os.startfile(analysed_video)

        else:
            self.drop_label.config(text="Invalid file type")
            self.prediction_label.config(text="Prediction: None")

    # Analyse button action
    def analyse_image(self):
        if self.image_path is None:
            self.drop_label.config(text="Drop an image first!")
            return

        # Crop face with YOLO
        output = process_image(
            self.image_path,
            self.face_model,
            conf_threshold=0.25,
            padding_ratio=0.15
        )

        # Handle crop output
        if isinstance(output, tuple):
            result_img, message = output
            self.drop_label.config(text=message)

        elif isinstance(output, str):
            self.drop_label.config(text=output)
            self.prediction_label.config(text="Prediction: None")
            return

        else:
            result_img = output

        # Transform cropped image exactly like validation/test
        input_tensor = res_val_transform(result_img).unsqueeze(0).to(DEVICE)

        # Predict with emotion classifier
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.softmax(outputs, dim=1)

            pred_idx = torch.argmax(probs, dim=1).item()
            conf = probs[0, pred_idx].item()

        pred_text = f"Prediction: {CLASS_NAMES[pred_idx]} ({conf:.3f})"
        print(pred_text)
        self.prediction_label.config(text=pred_text)

        # Display cropped image
        display_img = result_img.copy()
        display_img.thumbnail((400, 400))
        display_img = display_img.copy()

        self.tk_result = ImageTk.PhotoImage(display_img)

        self.result_image_label.config(image=self.tk_result)
        self.result_image_label.image = self.tk_result


# MAIN
if __name__ == "__main__":
    root = TkinterDnD.Tk()

    model_path = hf_hub_download(
        repo_id="arnabdhar/YOLOv8-Face-Detection",
        filename="model.pt"
    )

    face_model = YOLO(model_path)

    app = ImageDropApp(root, face_model)
    root.mainloop()

379 frames extracted!


Processing Video:   0%|          | 0/379 [00:00<?, ?it/s]

Window 1/379:   0%|          | 0/6 [00:00<?, ?it/s]

Window 2/379:   0%|          | 0/7 [00:00<?, ?it/s]

Window 3/379:   0%|          | 0/8 [00:00<?, ?it/s]

...

Window 377/379:   0%|          | 0/8 [00:00<?, ?it/s]

Window 378/379:   0%|          | 0/7 [00:00<?, ?it/s]

Window 379/379:   0%|          | 0/6 [00:00<?, ?it/s]


Adding audio back to video...
Moviepy - Building video processed_with_audio.mp4.
Moviepy - Writing video processed_with_audio.mp4



Moviepy - Done !
Moviepy - video ready processed_with_audio.mp4

Done!
Prediction: fear (0.620)
